# limpieza de datos

In [ ]:
import pandas as pd

from config import CRUDO, ANIOS

In [2]:
def leer_anio(anio):
    """Lee la base anual de importaciones de un año y la verifica contra el archivo de control.

    Supone que el zip ya se descomprimió en datos/crudo/imports_{anio}/
    y que los archivos se llaman impoaAA.csv e itotaAA.csv (AA = últimos dos dígitos del año).
    """
    carpeta = CRUDO / f"imports_{anio}"
    aa = str(anio)[-2:]

    datos = pd.read_csv(
        carpeta / f"impoa{aa}.csv",
        sep=";",
        decimal=",",
        encoding="latin1",
        dtype={"NCM": str, "Porg": str},
    )
    control = pd.read_csv(
        carpeta / f"itota{aa}.csv",
        sep=";",
        decimal=",",
        encoding="latin1",
    )

    # Entre 2019 y 2021 el INDEC usa "(u$s)" y desde 2022 "(USD)": unifico al formato nuevo
    datos.columns = datos.columns.str.replace("(u$s)", "(USD)", regex=False)
    control.columns = control.columns.str.replace("(u$s)", "(USD)", regex=False)

    # Verificación contra los totales de control del INDEC
    assert len(datos) == control.loc[0, "Total de registros"], f"{anio}: no coincide la cantidad de filas"
    assert abs(datos["CIF(USD)"].sum() - control.loc[0, "Total CIF(USD)"]) < 1, f"{anio}: no coincide el total CIF"

    return datos

for anio in ANIOS:
    aa = str(anio)[-2:]
    carpeta = CRUDO / f"imports_{anio}"
    datos_cols = pd.read_csv(carpeta / f"impoa{aa}.csv", sep=";", encoding="latin1", nrows=0).columns
    control_cols = pd.read_csv(carpeta / f"itota{aa}.csv", sep=";", encoding="latin1", nrows=0).columns
    print(anio)
    print("  datos:  ", list(datos_cols))
    print("  control:", list(control_cols))

2019
  datos:   ['Año', 'NCM', 'Porg', 'Pnet(kg)', 'FOB(u$s)', 'Flete(u$s)', 'Seguro(u$s)', 'CIF(u$s)']
  control: ['Tabla', 'Año', 'periodicidad', 'Total de registros', 'Total Pnet(kg)', 'Total FOB(u$s)', 'Total Flete(u$s)', 'Total Seguro(u$s)', 'Total CIF(u$s)', 'Unnamed: 9']
2020
  datos:   ['Año', 'NCM', 'Porg', 'Pnet(kg)', 'FOB(u$s)', 'Flete(u$s)', 'Seguro(u$s)', 'CIF(u$s)']
  control: ['Tabla', 'Año', 'periodicidad', 'Total de registros', 'Total Pnet(kg)', 'Total FOB(u$s)', 'Total Flete(u$s)', 'Total Seguro(u$s)', 'Total CIF(u$s)', 'Unnamed: 9']
2021
  datos:   ['Año', 'NCM', 'Porg', 'Pnet(kg)', 'FOB(u$s)', 'Flete(u$s)', 'Seguro(u$s)', 'CIF(u$s)']
  control: ['Tabla', 'Año', 'periodicidad', 'Total de registros', 'Total Pnet(kg)', 'Total FOB(u$s)', 'Total Flete(u$s)', 'Total Seguro(u$s)', 'Total CIF(u$s)', 'Unnamed: 9']
2022
  datos:   ['Año', 'NCM', 'Porg', 'Pnet(kg)', 'FOB(USD)', 'Flete(USD)', 'Seguro(USD)', 'CIF(USD)']
  control: ['Tabla', 'Año', 'periodicidad', 'Total de regis

In [3]:
bases = []
for anio in ANIOS:
    bases.append(leer_anio(anio))
    print(anio, "ok")

impo = pd.concat(bases, ignore_index=True)
print(impo.shape)
impo.head()

2019 ok
2020 ok
2021 ok
2022 ok
2023 ok
2024 ok
2025 ok
(490822, 8)


,Año,NCM,Porg,Pnet(kg),FOB(USD),Flete(USD),Seguro(USD),CIF(USD)
0,2019,01012100,200,9800.0,354774.00,45675.01,3636.86,404085.87
1,2019,01012100,203,11350.0,174203.39,19332.02,1683.94,195219.35
2,2019,01012100,208,11000.0,136291.00,6294.63,2725.82,145311.45
3,2019,01012100,212,8300.0,449989.50,46382.80,3226.60,499598.90
4,2019,01012100,218,500.0,35000.00,1724.64,367.25,37091.89


In [4]:
print(impo.dtypes)

assert set(impo["Año"]) == set(ANIOS), "Faltan o sobran años"
assert not impo.duplicated(["Año", "NCM", "Porg"]).any(), "Hay filas repetidas"
assert impo["CIF(USD)"].ge(0).all(), "Hay montos negativos"

Año              int64
NCM                str
Porg               str
Pnet(kg)       float64
FOB(USD)       float64
Flete(USD)     float64
Seguro(USD)    float64
CIF(USD)       float64
dtype: object


In [5]:
paises = pd.read_csv(
    CRUDO / "paises.csv",
    sep=";",
    dtype=str,
    keep_default_na=False,  # para que el código "NA" de Namibia no se lea como dato faltante
)
paises = paises.rename(columns={"_id": "Porg"})  # mismo nombre que en la base de importaciones

print(paises.shape)
paises.head()

(280, 3)


,Porg,iso2,nombre
0,301,AF,Afganistán
1,401,AL,Albania
2,404,AD,Andorra
3,149,AO,Angola
4,237,AG,Antigua y Barbuda


In [6]:
assert paises["Porg"].is_unique, "Hay códigos de país repetidos en el nomenclador"
assert (paises["iso2"] != "").all(), "Hay países sin código ISO"
assert set(impo["Porg"]).issubset(set(paises["Porg"])), "Hay códigos en la base que no están en el nomenclador"